# 第7周：函数参数进阶、异常处理（Day 43-49）

这是一个可以自动检测练习题的 Jupyter Notebook。配合《100天学习计划》第7周内容使用。

**这个notebook每一天都包含两部分：📖 知识点讲解（复习本周课程的讲解要点和实际场景）+ ✏️ 练习题与🔍自动检测。**

**怎么用这个notebook：**
1. 从上到下按顺序运行每一个cell（选中cell后按 `Shift + Enter`）
2. 看到 `✏️ 你的代码` 的cell，把 `pass` 删掉，写上你的代码
3. 写完之后，运行下面紧跟着的 `🔍 检测` cell，看看是不是显示 ✅（通过）
4. 如果显示 ❌，看看提示信息，回去改一改代码再重新运行这两个cell（先改代码的cell，再改检测的cell）

**一个小提醒：** 为了能让程序自动帮你检测，这周有些练习题会让你把结果 `return`（返回）出来，而不是直接 `print`（打印）——这其实是同一个知识点，只是写法上换了一种方式，之前学过的逻辑完全用得上。

这个notebook不会给你答案，只会告诉你做得对不对，加油！


## 📑 本周内容导航

每天的内容都是先有 **📖 知识点讲解**，再有 **✏️ 练习 + 🔍 检测**，请按顺序学习：

- Day 43：函数参数进阶——位置传参与关键字传参
- Day 44：可变参数 *args
- Day 45：异常处理基础（1）—— try/except
- Day 46：异常处理基础（2）——处理多种不同类型的异常
- Day 47：综合练习——把函数与异常处理结合起来
- Day 48：综合实战——搭建计算器程序框架
- Day 49：第7周 Checkpoint


In [ ]:
# 🔧 工具cell：检测练习题用的工具函数，不需要修改，直接运行一次就好
import contextlib
import io
import builtins

class mock_input:
    """检测需要用到input()的练习题时，临时按顺序提供输入内容"""
    def __init__(self, values):
        self._iter = iter(list(values))
        self._original = None
    def __enter__(self):
        self._original = builtins.input
        builtins.input = lambda prompt="": next(self._iter)
        return self
    def __exit__(self, *exc):
        builtins.input = self._original

def _values_match(result, expected, tol):
    if tol is not None:
        try:
            return abs(result - expected) <= tol
        except TypeError:
            return False
    return result == expected

def test_case(func, args=(), kwargs=None, expected=None, desc="", tol=None):
    kwargs = kwargs or {}
    try:
        result = func(*args, **kwargs)
    except Exception as e:
        print(f"❌ {desc} —— 运行出错：{type(e).__name__}: {e}")
        return False
    ok = _values_match(result, expected, tol)
    if ok:
        print(f"✅ {desc}")
    else:
        print(f"❌ {desc} —— 期望得到 {expected!r}，但实际得到 {result!r}")
    return ok

def test_case_with_input(func, inputs, expected=None, desc="", tol=None):
    try:
        with mock_input(inputs):
            result = func()
    except Exception as e:
        print(f"❌ {desc} —— 运行出错：{type(e).__name__}: {e}")
        return False
    ok = _values_match(result, expected, tol)
    if ok:
        print(f"✅ {desc}")
    else:
        print(f"❌ {desc} —— 期望得到 {expected!r}，但实际得到 {result!r}")
    return ok

def test_calculator_case(func, mock_values, expect_success, expect_value=None, desc="", tol=1e-6):
    buf = io.StringIO()
    crashed = False
    result = None
    err = ""
    try:
        with mock_input(mock_values):
            with contextlib.redirect_stdout(buf):
                result = func()
    except Exception as e:
        crashed = True
        err = f"{type(e).__name__}: {e}"
    output = buf.getvalue()
    if crashed:
        print(f"❌ {desc} —— 程序崩溃了：{err}")
        return False
    if expect_success:
        ok = (result is not None) and (abs(result - expect_value) <= tol)
        if ok:
            print(f"✅ {desc}（打印了：{output.strip()!r}）")
        else:
            print(f"❌ {desc} —— 期望算出 {expect_value}，但结果是 {result!r}（打印了：{output.strip()!r}）")
        return ok
    else:
        ok = len(output.strip()) > 0
        if ok:
            print(f"✅ {desc}（程序没有崩溃，打印了提示：{output.strip()!r}）")
        else:
            print(f"❌ {desc} —— 程序没有崩溃，但也没有打印任何提示信息")
        return ok

print("工具函数准备好了，可以开始做练习题了！")


## Day 43：函数参数进阶——位置传参与关键字传参

**学习目标**：理解调用函数时除了"按顺序传值"，还可以用"参数名=值"的方式传参，明白这样做能避免什么问题

### 📖 知识点讲解

**讲解要点**

- **位置参数**：按照函数定义的顺序依次传值，比如 `introduce("小明", 13, "篮球")`——第一个值自动对应第一个参数，第二个值自动对应第二个参数，顺序一旦搞错，结果就会跟着错。
  **实际场景**：就像填一张没有标注"姓名""年龄"字段名的表格，只能按固定顺序填；如果表格字段很多，很容易一不小心就把年龄填进了姓名的格子里。

- **关键字参数**：调用时用"参数名=值"的方式，比如 `introduce(name="小明", age=13, hobby="篮球")`，这样写不管参数顺序怎么变都不会出错，因为Python是靠"参数名"去对应，不是靠"位置"。
  **实际场景**：网购填写收货地址表单时，每个输入框都标注了字段名（收货人、电话、地址），不管你先填哪一个都不会出错，因为系统是靠"字段名"识别内容，不是靠"填写的先后顺序"——这正是关键字参数的思路。

- **两者最大的风险差别**：位置参数如果传错了顺序，**很多时候程序不会报错**，只是会悄悄算出一个错误的结果——这比直接报错更危险，因为不容易被发现。
  **实际场景**：想象一个"计算BMI"的函数，正常应该传入"体重"和"身高"，如果不小心把两者顺序传反了，程序照样能跑出一个数字，只是这个数字完全没有意义。


### ✏️ 练习1：`introduce(name, age, hobby)`

写一个函数，接收姓名、年龄、爱好，**返回**（return，不是print）这样格式的一句话：

`"我叫小明，今年13岁，爱好是篮球"`


In [ ]:
# ✏️ 你的代码
def introduce(name, age, hobby):
    pass


In [ ]:
# 🔍 检测
results_43_1 = []
results_43_1.append(test_case(introduce, ("小明", 13, "篮球"), expected="我叫小明，今年13岁，爱好是篮球", desc="用位置参数调用"))
results_43_1.append(test_case(introduce, (), {"name":"小红","age":12,"hobby":"跳舞"}, expected="我叫小红，今年12岁，爱好是跳舞", desc="用关键字参数调用"))
print(f"\n通过 {sum(results_43_1)}/{len(results_43_1)} 项")


### ✏️ 练习2：`order_pizza(size, topping="芝士")`

写一个函数，接收披萨尺寸，口味默认是"芝士"。返回这样格式的一句话：

`"一份大号的芝士披萨"`


In [ ]:
# ✏️ 你的代码
def order_pizza(size, topping="芝士"):
    pass


In [ ]:
# 🔍 检测
results_43_2 = []
results_43_2.append(test_case(order_pizza, ("大号",), expected="一份大号的芝士披萨", desc="不传topping，用默认值"))
results_43_2.append(test_case(order_pizza, (), {"topping":"培根","size":"中号"}, expected="一份中号的培根披萨", desc="用关键字参数，顺序写反也没问题"))
print(f"\n通过 {sum(results_43_2)}/{len(results_43_2)} 项")


### ✏️ 挑战：`calc_bmi(weight, height)`

写一个函数，计算并返回BMI（体重kg ÷ 身高m的平方）。

写完后可以自己试试：故意把weight和height的值传反调用一次，看看程序会不会报错——体会一下"参数顺序传错但程序没有报错"这种情况有多容易被忽略。


In [ ]:
# ✏️ 你的代码
def calc_bmi(weight, height):
    pass


In [ ]:
# 🔍 检测
results_43_3 = []
results_43_3.append(test_case(calc_bmi, (50, 1.6), expected=19.53, tol=0.01, desc="BMI计算 1"))
results_43_3.append(test_case(calc_bmi, (60, 1.8), expected=18.52, tol=0.01, desc="BMI计算 2"))
print(f"\n通过 {sum(results_43_3)}/{len(results_43_3)} 项")


## Day 44：可变参数 *args

**学习目标**：掌握用 `*args` 处理"数量不确定"的参数，理解它解决了什么实际问题

### 📖 知识点讲解

**讲解要点**

- 有时候不知道调用函数时会传入几个值，比如计算几个数字的总和，可能是2个，也可能是5个，这时候可以用 `def func(*args):` 让函数能接收**任意数量**的参数，这些参数会被自动打包成一个元组。
  **实际场景**：手机计算器App里的"连续加法"功能，用户可能只输入2个数字，也可能连续输入10个数字再求和，程序在写的时候不能提前限定"只能输入几个"。

- 函数内部可以像遍历列表、元组一样遍历 `args`，比如 `for num in args:`。
  **实际场景**：游戏结算时可能有2个人参与，也可能有6个人参与，需要把所有参与者的得分都加进总分里，用 `*args` 就不用为"最多几个人"设置限制。

- 调用的时候直接传入任意个普通的值就行，不需要自己先打包成列表或元组，Python会自动帮你处理。
  **实际场景**：就像吃自助餐一样，想拿几个菜就拿几个——`*args` 就是让函数也能"随便传几个都行"。


### ✏️ 练习1：`sum_all(*args)`

用for循环累加args里的所有数字并返回总和。


In [ ]:
# ✏️ 你的代码
def sum_all(*args):
    pass


In [ ]:
# 🔍 检测
results_44_1 = []
results_44_1.append(test_case(sum_all, (1,2), expected=3, desc="两个数字求和"))
results_44_1.append(test_case(sum_all, (1,2,3,4), expected=10, desc="四个数字求和"))
results_44_1.append(test_case(sum_all, (), expected=0, desc="没有参数时返回0"))
print(f"\n通过 {sum(results_44_1)}/{len(results_44_1)} 项")


### ✏️ 练习2：`collect_names(*args)`

把传进来的所有名字收集到一个列表里并返回（比如 `collect_names("小明","小红")` 要返回 `["小明", "小红"]`）。


In [ ]:
# ✏️ 你的代码
def collect_names(*args):
    pass


In [ ]:
# 🔍 检测
results_44_2 = []
results_44_2.append(test_case(collect_names, ("小明","小红"), expected=["小明","小红"], desc="收集两个名字"))
results_44_2.append(test_case(collect_names, (), expected=[], desc="没有名字时返回空列表"))
print(f"\n通过 {sum(results_44_2)}/{len(results_44_2)} 项")


### ✏️ 挑战：`find_max(*args)`

找出所有参数里最大的一个并返回（**不能用内置的 `max()`**，要自己写循环比较，可以回顾第3周Day19的思路）。


In [ ]:
# ✏️ 你的代码
def find_max(*args):
    pass


In [ ]:
# 🔍 检测
results_44_3 = []
results_44_3.append(test_case(find_max, (3,7,2,9,4), expected=9, desc="找最大值"))
results_44_3.append(test_case(find_max, (5,), expected=5, desc="只有一个数字"))
results_44_3.append(test_case(find_max, (-3,-7,-1), expected=-1, desc="全是负数"))
print(f"\n通过 {sum(results_44_3)}/{len(results_44_3)} 项")


## Day 45：异常处理基础（1）—— try/except

**学习目标**：理解程序为什么会"崩溃"，学会用 `try/except` 让程序遇到错误时不会直接停止运行

### 📖 知识点讲解

**讲解要点**

- 程序运行中遇到的"错误"在Python里叫**异常（Exception）**，比如把非数字文本转成int会报 `ValueError`，除以0会报 `ZeroDivisionError`。如果不处理，程序会直接终止运行。
  **实际场景**：手机银行App如果用户不小心在转账金额那里输错了内容（比如输入了文字而不是数字），App不应该直接闪退崩溃，而应该弹出"请输入正确的金额"这样的提示。

- **`try/except` 的基本结构**：把"可能出错"的代码放进`try`块，如果真的出错了，就会跳到对应的`except`块去执行，而不会让整个程序停止。
  **实际场景**：走钢丝表演下面通常会架一张安全网，`try`块里的代码就是"可能出问题的高风险动作"，`except`块就是"万一真的出问题了，安全网负责接住，不至于让整场表演（程序）崩掉"。

- 可以在`except`后面指定具体的错误类型，比如 `except ValueError:`，这样能针对不同的错误类型给出不同的、更准确的提示。
  **实际场景**：医院的分诊台会根据不同症状把病人分配到不同科室处理，而不是所有人都挤去同一个窗口；`except`指定具体错误类型也是同样"按类型分开处理"的思路。


### ✏️ 练习1：`convert_to_int(text)`

尝试把传入的文字转换成整数并返回；如果转换失败（`ValueError`），返回字符串 `"请输入有效的数字"`。


In [ ]:
# ✏️ 你的代码
def convert_to_int(text):
    pass


In [ ]:
# 🔍 检测
results_45_1 = []
results_45_1.append(test_case(convert_to_int, ("123",), expected=123, desc="正常数字文本"))
results_45_1.append(test_case(convert_to_int, ("abc",), expected="请输入有效的数字", desc="非数字文本"))
print(f"\n通过 {sum(results_45_1)}/{len(results_45_1)} 项")


### ✏️ 练习2：`divide_ten_by(divisor)`

计算并返回 `10 / divisor`；如果divisor是0（`ZeroDivisionError`），返回字符串 `"除数不能为0"`。


In [ ]:
# ✏️ 你的代码
def divide_ten_by(divisor):
    pass


In [ ]:
# 🔍 检测
results_45_2 = []
results_45_2.append(test_case(divide_ten_by, (2,), expected=5.0, desc="正常除法"))
results_45_2.append(test_case(divide_ten_by, (0,), expected="除数不能为0", desc="除数是0"))
print(f"\n通过 {sum(results_45_2)}/{len(results_45_2)} 项")


### ✏️ 挑战：`safe_divide_text(a, divisor_text)`

`a` 是一个数字，`divisor_text` 是用户输入的文字形式的除数。要求：
- 如果 `divisor_text` 不是合法数字，返回 `"请输入有效的数字"`
- 如果 `divisor_text` 转换后是0，返回 `"除数不能为0"`
- 否则返回 `a / divisor_text转换后的数字`

提示：想一想是不是需要写两个`except`块。


In [ ]:
# ✏️ 你的代码
def safe_divide_text(a, divisor_text):
    pass


In [ ]:
# 🔍 检测
results_45_3 = []
results_45_3.append(test_case(safe_divide_text, (10,"abc"), expected="请输入有效的数字", desc="除数不是数字"))
results_45_3.append(test_case(safe_divide_text, (10,"0"), expected="除数不能为0", desc="除数是0"))
results_45_3.append(test_case(safe_divide_text, (10,"2"), expected=5.0, desc="正常情况"))
print(f"\n通过 {sum(results_45_3)}/{len(results_45_3)} 项")


## Day 46：异常处理基础（2）——处理多种不同类型的异常

**学习目标**：掌握用多个 `except` 块分别处理不同类型的错误，写出对用户更友好的程序

### 📖 知识点讲解

**讲解要点**

- 一个 `try` 后面可以跟**多个** `except`，分别捕获不同类型的错误，程序会自动匹配到实际发生的那种错误对应的`except`块去执行。
  **实际场景**：网站的错误提示通常很具体——"用户名不存在"和"密码错误"是两条不同的提示，而不是笼统地说"登录失败"，这样用户才知道该怎么改。

- 如果不确定还会发生什么类型的错误，可以用 `except Exception as e:` 兜底捕获所有类型的错误（`e`里存着具体的错误信息），但更推荐尽量先明确写出已知会发生的错误类型。
  **实际场景**：保险公司通常会针对不同风险单独设计条款，但也常常会有一条"其他意外"的兜底条款，覆盖那些没有单独列出来的情况——`Exception`兜底就是类似的思路。

- 异常处理不是用来"掩盖bug"的，而是用来处理"用户操作可能出错"这种正常但意外的情况。
  **实际场景**：如果一个程序是因为自己的逻辑写错了才报错（比如变量名打错），这种应该直接改代码修复，而不是用`try/except`把这个错误"藏起来"不处理。


### ✏️ 练习1：`divide_two_texts(a_text, b_text)`

两个参数都是文字形式。要求：
- 任意一个不是合法数字，返回 `"请输入有效的数字"`
- 转换后除数是0，返回 `"除数不能为0"`
- 否则返回两个数相除的结果


In [ ]:
# ✏️ 你的代码
def divide_two_texts(a_text, b_text):
    pass


In [ ]:
# 🔍 检测
results_46_1 = []
results_46_1.append(test_case(divide_two_texts, ("10","abc"), expected="请输入有效的数字", desc="第二个不是数字"))
results_46_1.append(test_case(divide_two_texts, ("10","0"), expected="除数不能为0", desc="除数是0"))
results_46_1.append(test_case(divide_two_texts, ("10","2"), expected=5.0, desc="正常情况"))
print(f"\n通过 {sum(results_46_1)}/{len(results_46_1)} 项")


### ✏️ 练习2：`divide_with_fallback(a_text, b_text)`

在练习1的基础上，再加一个 `except Exception as e:` 作为兜底（处理其他没预料到的错误），其他行为保持一致。


In [ ]:
# ✏️ 你的代码
def divide_with_fallback(a_text, b_text):
    pass


In [ ]:
# 🔍 检测
import inspect

results_46_2 = []
results_46_2.append(test_case(divide_with_fallback, ("10","abc"), expected="请输入有效的数字", desc="第二个不是数字"))
results_46_2.append(test_case(divide_with_fallback, ("10","0"), expected="除数不能为0", desc="除数是0"))
results_46_2.append(test_case(divide_with_fallback, ("10","2"), expected=5.0, desc="正常情况"))

try:
    src = inspect.getsource(divide_with_fallback)
    has_fallback = "except Exception" in src
except Exception:
    has_fallback = False
if has_fallback:
    print("✅ 代码里包含了 except Exception 兜底写法")
else:
    print("⚠️ 提示：好像还没有在代码里加上 except Exception as e: 这个兜底块（这一项不影响上面的功能检测，但别忘了加上）")

print(f"\n通过 {sum(results_46_2)}/{len(results_46_2)} 项")


### ✏️ 挑战：`parse_guess(text)`

这是"安全版猜数字游戏"的核心小零件：尝试把用户输入的文字转换成数字并返回；如果转换失败，返回 `None`（不要让程序崩溃）。

写完这个函数之后，可以自己试着在一个 `while` 循环里调用它，模拟"一直让用户重新猜，直到输入合法数字为止"的完整效果——这部分不需要写在这里自动检测，自己在新的cell里试试看就好。


In [ ]:
# ✏️ 你的代码
def parse_guess(text):
    pass


In [ ]:
# 🔍 检测
results_46_3 = []
results_46_3.append(test_case(parse_guess, ("50",), expected=50, desc="合法数字"))
results_46_3.append(test_case(parse_guess, ("abc",), expected=None, desc="非法输入返回None"))
print(f"\n通过 {sum(results_46_3)}/{len(results_46_3)} 项")


## Day 47：综合练习——把函数与异常处理结合起来

**学习目标**：学会把"容易出错的逻辑"包装成一个安全的函数，让主程序调用起来更简洁可靠

### 📖 知识点讲解

**讲解要点**

- 可以把 `try/except` 直接写在函数内部，让函数自己处理错误，然后返回一个"结果"或者一个"错误提示"给调用它的地方，这样主程序就不用每次调用都重复写一遍`try/except`。
  **实际场景**：很多手机App的"网络请求"背后都封装了统一的错误处理函数，不管是App里哪个页面发起网络请求，出错时都能得到一致、写得很清楚的提示。

- 函数如果内部已经处理了异常，通常会返回一个特殊值（比如 `None`）或者一段提示文字，代表"这次调用没有成功"，调用它的地方需要检查这个返回结果再决定下一步怎么做。
  **实际场景**：自动取款机如果余额不足，不会直接吐钞然后死机，而是返回一个"余额不足"的提示，让整个流程能继续走下去。


### ✏️ 练习1：`safe_int(text)`

尝试转换成整数并返回；转换失败就返回 `None`。


In [ ]:
# ✏️ 你的代码
def safe_int(text):
    pass


In [ ]:
# 🔍 检测
results_47_1 = []
results_47_1.append(test_case(safe_int, ("123",), expected=123, desc="正常数字"))
results_47_1.append(test_case(safe_int, ("abc",), expected=None, desc="非数字返回None"))
print(f"\n通过 {sum(results_47_1)}/{len(results_47_1)} 项")


### ✏️ 练习2：`safe_divide(a, b)`

尝试计算a除以b并返回；如果b是0，返回字符串 `"无法除以0"`。


In [ ]:
# ✏️ 你的代码
def safe_divide(a, b):
    pass


In [ ]:
# 🔍 检测
results_47_2 = []
results_47_2.append(test_case(safe_divide, (10,2), expected=5.0, desc="正常除法"))
results_47_2.append(test_case(safe_divide, (10,0), expected="无法除以0", desc="除以0"))
print(f"\n通过 {sum(results_47_2)}/{len(results_47_2)} 项")


### ✏️ 挑战：`safe_calculate(a_text, b_text)`

结合上面两个函数：先用 `safe_int()` 把两个文本都转换成数字，如果任意一个转换失败（结果是`None`），返回 `"请输入有效的数字"`；否则用 `safe_divide()` 计算结果并返回。


In [ ]:
# ✏️ 你的代码
# 提示：可以在这里调用你前面写好的 safe_int 和 safe_divide
def safe_calculate(a_text, b_text):
    pass


In [ ]:
# 🔍 检测
results_47_3 = []
results_47_3.append(test_case(safe_calculate, ("10","2"), expected=5.0, desc="两个都合法"))
results_47_3.append(test_case(safe_calculate, ("abc","2"), expected="请输入有效的数字", desc="第一个不合法"))
results_47_3.append(test_case(safe_calculate, ("10","0"), expected="无法除以0", desc="除数是0"))
print(f"\n通过 {sum(results_47_3)}/{len(results_47_3)} 项")


## Day 48：综合实战——搭建计算器程序框架

**学习目标**：为明天的Checkpoint做准备，先把"计算器"程序的整体结构搭好（这一步先不用管异常处理）

### 📖 知识点讲解

**讲解要点**

- 今天的目标是先让计算器在"正常输入"的情况下能跑起来，明天再专门加上异常处理——这是很多实际项目开发时的顺序：先把主要功能跑通，再逐步加固对"边界情况"的处理。
  **实际场景**：很多软件的开发过程也是这样，先做出一个"能用的版本"，上线测试后再根据用户反馈逐步加上各种异常情况的处理。

- 可以把"读取运算符"、"根据运算符计算结果"这两部分分别写成函数，让代码结构更清晰，方便明天往里加异常处理。
  **实际场景**：这跟第6周搭"通讯录管理系统"时的思路是一样的——先把整体功能拆成几个各自独立的函数，一块一块搭建和测试。


### ✏️ 练习1：`get_operator()`

用 `input()` 询问用户想做哪种运算（+、-、*、/），并返回这个符号。


In [ ]:
# ✏️ 你的代码
def get_operator():
    pass


In [ ]:
# 🔍 检测（会自动模拟输入，不需要你真的手动输入）
results_48_1 = []
results_48_1.append(test_case_with_input(get_operator, ["+"], expected="+", desc="模拟输入 +"))
results_48_1.append(test_case_with_input(get_operator, ["*"], expected="*", desc="模拟输入 *"))
print(f"\n通过 {sum(results_48_1)}/{len(results_48_1)} 项")


### ✏️ 练习2：`calculate(a, b, operator)`

根据传入的运算符（+ - * /）做对应的计算并返回结果（用if-elif处理这4种运算符）。


In [ ]:
# ✏️ 你的代码
def calculate(a, b, operator):
    pass


In [ ]:
# 🔍 检测
results_48_2 = []
results_48_2.append(test_case(calculate, (10,5,"+"), expected=15, desc="加法"))
results_48_2.append(test_case(calculate, (10,5,"-"), expected=5, desc="减法"))
results_48_2.append(test_case(calculate, (10,5,"*"), expected=50, desc="乘法"))
results_48_2.append(test_case(calculate, (10,5,"/"), expected=2.0, desc="除法"))
print(f"\n通过 {sum(results_48_2)}/{len(results_48_2)} 项")


### ✏️ 挑战：`run_calculator()`

把上面两个函数和获取两个数字的 `input()` 结合起来，写出一个完整能运行的计算器主程序：
- 依次用 `input()` 获取"第一个数字"、运算符、"第二个数字"（两个数字都要转换成float）
- 用 `calculate()` 计算结果
- 打印 `f"结果是：{结果}"`
- 最后把结果 `return` 出去

这一步先不用加任何异常处理，只要正常输入的情况下能算对结果就行，为明天的Checkpoint打基础。


In [ ]:
# ✏️ 你的代码
def run_calculator():
    pass


In [ ]:
# 🔍 检测（会自动模拟输入，不需要你真的手动输入）
results_48_3 = []
results_48_3.append(test_case_with_input(run_calculator, ["10","+","5"], expected=15.0, desc="10+5"))
results_48_3.append(test_case_with_input(run_calculator, ["20","/","4"], expected=5.0, desc="20/4"))
print(f"\n通过 {sum(results_48_3)}/{len(results_48_3)} 项")


## Day 49：第7周 Checkpoint

### 小测验（口头回答，答案由家长核对，这里不自动检测）

1. 位置参数和关键字参数调用函数时，最大的区别是什么？
2. `*args` 在函数定义时的作用是什么？它在函数内部会是什么数据类型？
3. `try/except` 的作用是什么？如果不用它，程序遇到错误会发生什么？
4. 一个 `try` 后面可以跟多个 `except` 吗？这样做的好处是什么？
5. 为什么说异常处理不应该被用来"掩盖bug"？


### ✏️ 挑战题：给计算器加上异常处理 —— `run_calculator_safe()`

在Day48的 `run_calculator()` 基础上，加上异常处理：
- 复用Day48的逻辑（获取两个数字和运算符、计算、打印并返回结果）
- 用 `try/except` 处理两种情况：**输入的不是数字**（`ValueError`），和**除数是0**（`ZeroDivisionError`）
- 两种情况都要 `print` 一条给用户看的清晰提示，而不是让程序崩溃
- 处理完错误后，函数应该正常结束（不要再把错误抛出去）

**完成标准**
- [ ] 程序遇到错误输入不会崩溃
- [ ] 至少用try/except处理2种不同错误
- [ ] 有对用户友好的错误提示


In [ ]:
# ✏️ 你的代码
def run_calculator_safe():
    pass


In [ ]:
# 🔍 检测（会自动模拟输入，不需要你真的手动输入）
results_49 = []
results_49.append(test_calculator_case(run_calculator_safe, ["10","/","2"], True, 5.0, desc="正常情况：10/2"))
results_49.append(test_calculator_case(run_calculator_safe, ["10","/","0"], False, desc="除数是0，不应该崩溃，要有提示"))
results_49.append(test_calculator_case(run_calculator_safe, ["abc","+","5"], False, desc="输入非数字，不应该崩溃，要有提示"))

print()
passed = sum(results_49)
total = len(results_49)
print(f"第7周Checkpoint：通过 {passed}/{total} 项")
if passed == total:
    print("🎉 全部通过！可以让家长看看小测验的口头回答，然后进入第8周啦。")


---

恭喜完成第7周的所有练习！如果有检测没通过，可以回去看看对应的"讲解要点"和"实际场景"部分，想一想问题出在哪里——记得，调试报错信息本身就是很重要的编程能力。
